# 4. SHAP : comment le Random Forest détecte une attaque

**Objectif.** Expliquer les décisions du modèle retenu. SHAP attribue à chaque feature sa part
dans la décision, ligne par ligne. On regarde d'abord l'ensemble des attaques, puis chaque
scénario, puis une ligne précise.

In [ ]:
%%capture
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "backend" / "scripts")]
import build_dataset as bd
import transmission_real as tr
import transmission_models as tm
import transmission_study as ts
from app.api.v1 import phase2
from app.core import paths

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 3.5)
SEED = tr.SEED
COLS = {"weather": "temperature", "fridge": "fridge_temperature", "thermostat": "current_temperature"}
EPISODES_PATH = paths.PROCESSED_DIR / "episodes_transmission_reel.json"
MODEL_DIR = paths.MODELS_DIR / "transmission_reel"

In [ ]:
import joblib, shap

df = pd.read_csv(tr.DATASET_PATH, parse_dates=["datetime"], keep_default_na=False)
df["garde"] = df["garde"].astype(str).str.lower().eq("true")
df, feats = tm.engineer(df)
rf = joblib.load(MODEL_DIR / "random_forest.pkl")

y = df["label"].to_numpy().astype(int)
# Découpage recalculé (non stocké dans le dataset), non aléatoire 70/15/15.
split = tr.block_split(len(df), run=0)
types = df["attack_type"].to_numpy()
hors_train = np.where(np.isin(split, ["val", "test"]) & (y == 1) & ~df["garde"].to_numpy())[0]
rng = np.random.default_rng(SEED)
rows = np.sort(rng.choice(hors_train, min(2000, len(hors_train)), replace=False))
X = df[feats].to_numpy(float)
print(f"{len(rows):,} lignes d'attaque de validation et de test (jamais vues à l'entraînement)")


In [ ]:
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X[rows])
if isinstance(sv, list):
    sv = sv[-1]
if sv.ndim == 3:
    sv = sv[:, :, -1]          # contribution à la classe « attaque »
print("valeurs SHAP :", sv.shape)

## 4.1 Vue d'ensemble : quelles features font détecter ?

In [ ]:
shap.summary_plot(sv, X[rows], feature_names=feats, plot_type="bar", max_display=12, show=False)
plt.title("Importance moyenne |SHAP| sur les lignes d'attaque"); plt.tight_layout(); plt.show()

In [ ]:
shap.summary_plot(sv, X[rows], feature_names=feats, max_display=12, show=False)
plt.tight_layout(); plt.show()

Lecture : chaque point est une ligne d'attaque ; à droite, la feature pousse vers « attaque ».
Un **écart-type sur 1 minute très bas** (capteur figé ou valeurs forgées trop régulières) et une
**moyenne lissée qui baisse** (biais) sont les premiers signaux. Le drapeau `fridge_repeat_2min`
porte à lui seul la détection des rejeus.

## 4.2 Par scénario

In [ ]:
imp = np.abs(sv)
tab = {}
for sc in tr.ATTACK_NAMES:
    m = types[rows] == sc
    if m.any():
        moy = imp[m].mean(axis=0)
        top = np.argsort(moy)[::-1][:4]
        tab[sc] = {f"feature {i+1}": f"{feats[j]} ({moy[j]:.3f})" for i, j in enumerate(top)}
pd.DataFrame(tab).T

In [ ]:
fichier = MODEL_DIR / "shap_transmission_reel.json"
if fichier.exists():
    cumul = json.load(open(fichier, encoding="utf-8"))
    print(f"Même analyse cumulée sur les 9 répétitions du script ({cumul['n_lignes']:,} lignes) :")
    display(pd.DataFrame({sc: {f"feature {i+1}": f"{f['feature']} ({f['importance']:.3f})" for i, f in enumerate(lst[:4])}
                          for sc, lst in cumul["par_scenario"].items()}).T)

| Scénario | Feature qui fait détecter | Ce qu'elle capte |
| --- | --- | --- |
| Gel (frigo ou thermostat) | écart-type sur 1 minute du capteur | la valeur ne bouge plus |
| Injection nominale | écart-type sur 1 et 5 minutes du frigo | les valeurs forgées sont trop régulières |
| Rejeu | `fridge_repeat_2min` | les 2 dernières minutes ont déjà été transmises |
| Biais de −2 °C, dérive, biais du thermostat | moyenne lissée 10 min, moyenne 5 min | le niveau moyen a baissé |

## 4.3 Une ligne à la fois : pourquoi cette alerte ?

In [ ]:
def expliquer(scenario, k=8):
    cand = [i for i, r in enumerate(rows) if types[r] == scenario]
    if not cand:
        print(scenario, ": aucune ligne hors entraînement dans cette répétition"); return
    i = cand[len(cand) // 2]
    contrib = pd.Series(sv[i], index=feats).sort_values(key=np.abs, ascending=False)[:k][::-1]
    plt.figure(figsize=(8, 3.2))
    plt.barh(contrib.index, contrib.values, color=np.where(contrib.values > 0, "#d9534f", "#5b9bd5"))
    plt.axvline(0, color="k", lw=0.8)
    plt.title(f"{scenario} — ligne {rows[i]} : probabilité d'attaque {rf.predict_proba(X[[rows[i]]])[0, 1]:.2f}")
    plt.xlabel("contribution SHAP (rouge : pousse vers « attaque »)"); plt.tight_layout(); plt.show()

for sc in ("MITM_gel", "MITM_rejeu", "Injection_nominale", "Injection_biais_2.0", "MITM_gel_thermostat"):
    expliquer(sc)

Chaque alerte peut ainsi être justifiée à l'opérateur par deux ou trois features lisibles :
« l'écart-type du frigo sur 1 minute est tombé à 0 », « les 2 dernières minutes avaient déjà été
transmises », « la moyenne lissée du frigo a baissé de 2 °C ». C'est la base de la partie
**recommandation** (rapport pour l'opérateur).